# TRACE × DP-Fusion — the whole pipeline

The two halves, joined. `fusit.trace` decides which characters leak an attribute;
`fusit.dp_fusion` bounds how much those characters can move the output. This notebook runs
them end to end and then attacks the result.

The reason to join them is a gap in DP-Fusion as published. Its guarantee covers exactly the
spans the tagger marked, and its tagger marks **named entities**. That is the right target
when the secret is written down, and the wrong one when it is *inferable*: nobody in the
document below states their occupation, yet an attacker reads it off in one shot. There is no
entity to tag, so a NER-based pipeline protects nothing and reports ε as though it had.

Three roles, three models — each the one the respective line of work uses:

| stage | role | model |
| --- | --- | --- |
| 1 | cue tagging: attention (`V_att`) and inference chain (`V_cot`) | Llama-2-7B-Chat |
| 2 | DP-Fusion paraphrase | Qwen2.5-7B-Instruct (the paper's model) |
| 3 | attack / evaluation | Llama-3.1-8B-Instruct |

Keeping the attacker distinct from the tagger matters for reading the result: if the same
model both picked the cues and then tried to recover the attribute, a drop in attack success
could just mean the defence removed exactly what *that* model looks at.

None of these fit on a 24 GB card together, so each stage **loads its model, runs, and frees
it** before the next. Stage 2 also generates the αβ sweep, so stage 3 can attack every output
with a single load.

    D ─▶ [Llama-2] CueTagger ─▶ X_priv ─▶ groups ─▶ [Qwen] DP-Fusion ─▶ paraphrases ─▶ [Llama-3.1] attack

**Kernel**: `Python (dpfusion-repro)`. **Hardware**: ~16 GB free for the largest stage.

## 0. Setup

In [ ]:
import gc, math, textwrap, time
from collections import Counter

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from fusit.dataset import Span, get_dataset
from fusit.dp_fusion import build_contexts, dp_fusion_groups_incremental
from fusit.trace import CueTagger, guess_attribute, merge_spans, redact

TAGGER_ID     = "NousResearch/Llama-2-7b-chat-hf"          # V_att and V_cot
PARAPHRASE_ID = "Qwen/Qwen2.5-7B-Instruct"                 # DP-Fusion generation
ATTACKER_ID   = "NousResearch/Meta-Llama-3.1-8B-Instruct"  # evaluation

DATASET = "synthetic"          # implicit PII -- one comment, one target attribute
ITEM_N  = 0
SOURCES = ["ner", "cot", "att"]
K_ATT   = 10

ALPHA   = 2.0
DELTA   = 1e-3
MAX_DIV = 0.10                 # the cap, i.e. the paper's alpha*beta
SWEEP   = (0.01, 0.10, 1.00)   # generated in stage 2, attacked in stage 3
MAX_NEW = 120

print(f"torch {torch.__version__} | cuda {torch.cuda.is_available()}")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"{torch.cuda.get_device_name(0)}: {free/1e9:.1f} of {total/1e9:.1f} GB free")

In [ ]:
def load(model_id):
    t0 = time.perf_counter()
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="cuda:0")
    mdl.eval()
    print(f"loaded {model_id} in {time.perf_counter()-t0:.0f}s | "
          f"{torch.cuda.memory_allocated()/1e9:.1f} GB allocated")
    return mdl, tok


def release(*names):
    """Free a stage's model. Deleting the local names is not enough on its own -- anything
    still bound in the notebook namespace keeps the weights alive -- so drop them from
    globals, collect, and confirm the memory actually came back."""
    for n in names:
        globals().pop(n, None)
    gc.collect()
    torch.cuda.empty_cache()
    print(f"released {names} | {torch.cuda.memory_allocated()/1e9:.2f} GB still allocated")


item = get_dataset(DATASET).select(n=8, seed=0)[ITEM_N]
TEXT = item.text
ATTRIBUTE = next(iter(item.relevant_pii))
TRUTH = item.relevant_pii[ATTRIBUTE]

print(f"item      : {item.username}")
print(f"attribute : {ATTRIBUTE}")
print(f"truth     : {TRUTH!r}")
print(f"hardness  : {getattr(item, 'hardness', '-')}  (1 = stated outright, 5 = oblique)\n")
print(textwrap.fill(TEXT, 100))

## Stage 1 — cue tagging with Llama-2-7B-Chat

`CueTagger.explain` returns the spans **per source** rather than the union, which is the whole
subject here: the union alone cannot tell you whether NER contributed anything.

Llama-2 needs two things `fusit.trace.chat` already handles: its tokenizer ships no chat
template (so Meta's `[INST] <<SYS>>` format is used), and its context is 4096 tokens.

In [ ]:
tag_model, tag_tok = load(TAGGER_ID)

tagger = CueTagger(tag_model, tag_tok, attributes=[ATTRIBUTE], sources=SOURCES, k=K_ATT)
print(tagger)

t0 = time.perf_counter()
by_source = tagger.explain(TEXT)
print(f"tagged in {time.perf_counter()-t0:.1f}s\n")

def pct(spans):
    return 100 * sum(e - s for s, e in merge_spans(spans)) / len(TEXT)

for src in SOURCES:
    sp = by_source[src]
    print(f"{src:4s} {len(sp):3d} spans {pct(sp):5.1f}%  {str([TEXT[s:e] for s, e in sp])[:70]}")

X_PRIV = merge_spans([s for sp in by_source.values() for s in sp])
print(f"\nX_priv {len(X_PRIV):3d} spans {pct(X_PRIV):5.1f}% of the document")

del tagger
release("tag_model", "tag_tok")

In [ ]:
def bracket(text, spans, width=100):
    out, prev = [], 0
    for s, e in merge_spans(spans):
        out.append(text[prev:s]); out.append("[" + text[s:e] + "]"); prev = e
    out.append(text[prev:])
    return textwrap.fill("".join(out), width)

print("--- NER(D) alone: what DP-Fusion's original tagger would protect ---")
print(bracket(TEXT, by_source["ner"]) if by_source["ner"] else "(nothing)")
print("\n--- X_priv: NER u V_cot u V_att ---")
print(bracket(TEXT, X_PRIV))

### Cue spans become privacy groups

Algorithm 1 wants a partition of the private tokens. Entity types are how the paper happens to
partition them on TAB-ECHR; here the natural axis is **which signal found the span**, so each
source becomes a group with its own λ, divergence trace and ε — the NER entities, the chain's
quoted evidence and the attention words need not share one budget.

The mechanism assumes "the tagger assigns every sensitive token to exactly one privacy group",
so overlaps are resolved by precedence `ner > cot > att`: entities are the least ambiguous
evidence and keep the span they share.

In [ ]:
def to_groups(by_source, precedence=("ner", "cot", "att")):
    claimed, out = set(), []
    for src in precedence:
        for s, e in merge_spans(by_source.get(src, [])):
            free = sorted(set(range(s, e)) - claimed)
            if not free:
                continue
            claimed.update(free)
            start = prev = free[0]
            for i in free[1:] + [None]:
                if i is None or i != prev + 1:
                    out.append(Span(start, prev + 1, src, TEXT[start:prev + 1]))
                    start = i
                if i is not None:
                    prev = i
    return sorted(out, key=lambda s: s.start)

GROUP_SPANS = to_groups(by_source)
for src in SOURCES:
    owned = [s for s in GROUP_SPANS if s.entity_type == src]
    chars = sum(s.end - s.start for s in owned)
    print(f"{src:4s} owns {len(owned):3d} spans, {chars:4d} chars ({100*chars/len(TEXT):5.1f}%)")

covered = {i for s in GROUP_SPANS for i in range(s.start, s.end)}
assert covered == {i for s, e in X_PRIV for i in range(s, e)}, "partition must cover exactly X_priv"
assert sum(s.end - s.start for s in GROUP_SPANS) == len(covered), "a character sits in two groups"
print(f"\npartition covers exactly X_priv ({len(covered)} chars), no character in two groups")

## Stage 2 — DP-Fusion paraphrase with Qwen2.5-7B-Instruct

The contexts are built with **this** stage's tokenizer: token alignment is a property of one
tokenization, so building them with Llama-2's tokenizer and generating with Qwen would be
meaningless.

The continuation is sliced off the token ids. `dp_fusion_groups_incremental` returns the
decoded *whole* PUBLIC sequence and mutates `token_ids_groups` in place, so string-length
arithmetic on the return value is fragile.

In [ ]:
gen_model, gen_tok = load(PARAPHRASE_ID)

ctx = build_contexts(gen_tok, TEXT, GROUP_SPANS, entity_types=SOURCES)
GROUPS = [k for k in ctx if k != "PUBLIC"]      # sources with no spans get no group
prompt_len = len(ctx["PUBLIC"])
assert len({len(v) for v in ctx.values()}) == 1
print("groups:", GROUPS, "| every context", prompt_len, "tokens")
for g in GROUPS:
    print(f"  {g:4s} reveals {sum(a != b for a, b in zip(ctx['PUBLIC'], ctx[g])):3d} tokens PUBLIC hides")


def paraphrase(cap):
    io = {k: torch.tensor(v) for k, v in ctx.items()}
    _, lam, div = dp_fusion_groups_incremental(
        token_ids_groups=io, beta_dict={g: cap for g in GROUPS}, alpha=ALPHA,
        model=gen_model, tokenizer=gen_tok, temperature=1.0, max_new_tokens=MAX_NEW,
    )
    text = gen_tok.decode(io["PUBLIC"][prompt_len:].tolist(), skip_special_tokens=True)
    assert all(0 <= d <= cap + 1e-9 for g in GROUPS for d in div[g]), f"cap {cap} violated"
    return text, lam, div


t0 = time.perf_counter()
OUTPUTS = {cap: paraphrase(cap) for cap in sorted(set(SWEEP) | {MAX_DIV})}
print(f"\n{len(OUTPUTS)} paraphrases in {time.perf_counter()-t0:.1f}s\n")

PARAPHRASE, lam_hist, div_hist = OUTPUTS[MAX_DIV]
print(f"--- alpha*beta = {MAX_DIV} ---")
print(textwrap.fill(PARAPHRASE.strip(), 100))

release("gen_model", "gen_tok")

In [ ]:
T, m = len(next(iter(div_hist.values()))), len(GROUPS)

def theorem4(betas, m=m):
    per = sum((1/(ALPHA-1)) * math.log((m-1)/m + (1/m)*math.exp((ALPHA-1)*4*b)) for b in betas)
    return per + math.log(1/DELTA)/(ALPHA-1)

print(f"alpha*beta = {MAX_DIV}, T = {T}, m = {m}, alpha = {ALPHA}, delta = {DELTA}\n")
print(f"{'group':>5} | {'lambda mean':>11} {'min':>7} | {'div mean':>9} {'max':>7} | {'epsilon':>7}")
print("-" * 60)
for g in GROUPS:
    L, D = lam_hist[g], div_hist[g]
    print(f"{g:>5} | {sum(L)/len(L):11.3f} {min(L):7.3f} | {sum(D)/len(D):9.4f} {max(D):7.4f} | "
          f"{theorem4([d/ALPHA for d in D]):7.3f}")

## Stage 3 — attack with Llama-3.1-8B-Instruct

The same adversarial-inference prompt `V_cot` was built on, run by a *different* model against
each output. First the defences at the default budget, then the sweep.

Scoring is deliberately crude — substring containment either way — where
`repro/synthpai_eval.py` buckets categorical attributes properly. Good enough to separate
"named it" from "did not"; not good enough to report as an ASR.

In [ ]:
atk_model, atk_tok = load(ATTACKER_ID)

def hit(guesses, truth):
    t = truth.lower()
    return any(t in g.lower() or g.lower() in t for g in guesses)

def attack(text):
    if not text.strip():
        return []
    return guess_attribute(text, ATTRIBUTE, atk_model, atk_tok)["guesses"][:3]

conditions = {
    "no defense":           TEXT,
    "NER redaction":        redact(TEXT, by_source["ner"]) if by_source["ner"] else TEXT,
    "X_priv redaction":     redact(TEXT, X_PRIV),
    "DP-Fusion paraphrase": PARAPHRASE,
}

print(f"truth: {TRUTH!r}\n")
print(f"{'condition':<22} {'hit':>4}  top-3 guesses")
print("-" * 92)
results = {}
for name, txt in conditions.items():
    g = attack(txt)
    results[name] = hit(g, TRUTH)
    print(f"{name:<22} {'YES' if results[name] else 'no':>4}  {g}")

# A defence can only be read against an attacker that succeeds undefended. If the baseline
# misses, every row below it means "the attacker failed", not "the defence worked".
if not results["no defense"]:
    print("\nWARNING: the attacker misses on the UNDEFENDED text, so this table shows nothing")
    print("about the defence. Use a stronger attacker, or an item it can actually solve.")
else:
    stopped = [n for n, h in results.items() if n != "no defense" and not h]
    print(f"\nbaseline recovers the attribute; defences that stopped it: {stopped or 'none'}")

In [ ]:
print(f"{'alpha*beta':>11} {'lambda':>7} {'max eps':>8}  attacker's top-3   (truth: {TRUTH!r})")
print("-" * 92)
for cap in SWEEP:
    text, lh, dh = OUTPUTS[cap]
    lam = sum(v for vs in lh.values() for v in vs) / sum(len(v) for v in lh.values())
    eps = max(theorem4([d/ALPHA for d in dh[g]]) for g in GROUPS)
    g3 = attack(text)
    print(f"{cap:11.2f} {lam:7.3f} {eps:8.2f}  {'HIT ' if hit(g3, TRUTH) else '    '}{g3}")

print("\nRead this column against the 'no defense' row above, not against zero.")

release("atk_model", "atk_tok")

## Scratch

`by_source`, `GROUP_SPANS`, `OUTPUTS`, `lam_hist`, `div_hist` survive; the models do not —
each stage freed its own, so reload with `load(...)` to poke at one.

Worth trying from here:

- `DATASET = "synthpai"` — profiles instead of single comments, several attributes at once, so
  V_cot runs per attribute and X_priv grows accordingly,
- give the sources different budgets in `paraphrase` (e.g. `{"ner": 0.01, "cot": 0.1,
  "att": 0.1}`) — the point of partitioning by source is that they need not share one ε,
- swap `TAGGER_ID` and `ATTACKER_ID` to see how much of the defence is specific to the model
  that chose the cues,
- compare against `random_spans_matched` from `fusit.trace`: a coverage-matched random control
  is the only way to tell "the cue picked the risky words" from "more text was deleted".